In [1]:
#Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Spark Introduction")
    .master("local[*]")
    .getOrCreate()
)

In [2]:
data = spark.read.format("csv").option("header", True).option("inferSchema", True).option("mode", "PERMISSIVE").load("netflix_titles.csv")

In [3]:
data.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan

In [4]:
#Count null / NaN values per column

from pyspark.sql.functions import col, sum, isnan, when

data.select([
    sum(when(col(c).isNull() | isnan(col(c)), 1).otherwise(0)).alias(c)
    for c in data.columns]).show()


TypeError: 'JavaPackage' object is not callable

In [5]:
emp_data = [
    ["001","101","John Doe","30","Male","HR","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","IT","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","Sales","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","Sales","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","IT","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","HR","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","Marketing","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","Sales","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","HR","80000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","IT","47000","2018-08-01"],
    ["011","104","David Park","38","Male","Sales","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","HR", "54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","HR","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","IT","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","Sales","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","Marketing","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","Marketing","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","Marketing","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","Sales","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","Marketing","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string,dept string ,salary string, hire_date string"
df = spark.createDataFrame(data = emp_data, schema = emp_schema)
df.show()

+-----------+-------------+-------------+---+------+---------+------+----------+
|employee_id|department_id|         name|age|gender|     dept|salary| hire_date|
+-----------+-------------+-------------+---+------+---------+------+----------+
|        001|          101|     John Doe| 30|  Male|       HR| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|Female|       IT| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|  Male|    Sales| 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|Female|    Sales| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male|       IT| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|Female|       HR| 52000|2018-07-01|
|        007|          101|James Johnson| 42|  Male|Marketing| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|Female|    Sales| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male|       HR| 80000|2016-06-01|
|        010|          104| 

In [6]:
#Filter rows with null in a specific column
data.filter(col("director").isNull() | isnan(col("director"))).show()

TypeError: 'JavaPackage' object is not callable

In [8]:
#Get total number of missing values in the whole DataFrame
from pyspark.sql.functions import col, sum, isnan, when

missing_count = data.select([
    sum(when(col(c).isNull() | isnan(col(c)), 1).otherwise(0))
    for c in data.columns
]).rdd.flatMap(lambda x: x).sum()

print(f"Total missing values: {missing_count}")


Total missing values: 4329


In [11]:
# Replace all nulls with a specific value (e.g., 0)

df_filled = data.fillna(0)
df_filled.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan

In [16]:
# Replace nulls with different types (string vs numeric)

df_filled = data.select(col("director")).fillna("Unknown")
df_filled.show()


+--------------------+
|            director|
+--------------------+
|     Kirsten Johnson|
|             Unknown|
|     Julien Leclercq|
|             Unknown|
|             Unknown|
|       Mike Flanagan|
|Robert Cullen, Jo...|
|        Haile Gerima|
|     Andy Devonshire|
|      Theodore Melfi|
|             Unknown|
|   Kongkiat Komesiri|
| Christian Schwochow|
|       Bruno Garotti|
|             Unknown|
|             Unknown|
|Pedro de Echave G...|
|             Unknown|
|          Adam Salky|
|             Unknown|
+--------------------+
only showing top 20 rows



In [ ]:
#Replace null values in a specific column

df_filled = df.fillna({"age": 0})


In [20]:
#Replace null using na.fill() (alias for fillna)

df_filled = df.na.fill("missing")     # replace nulls in all string cols
df_filled = df.na.fill(0)             # replace nulls in all numeric cols
df_filled = df.na.fill({"city": "Unknown", "salary": 0})


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `city` cannot be resolved. Did you mean one of the following? [`employee_id`, `department_id`, `name`, `age`, `gender`, `dept`, `salary`, `hire_date`].

In [21]:
#Replace null values with a computed value
#Sometimes you want to replace null with a mean, median, or mode:

from pyspark.sql.functions import col, mean

# Example: replace nulls in "salary" with mean salary
mean_val = df.select(mean(col("salary"))).collect()[0][0]

df_filled = df.na.fill({"salary": mean_val})


In [ ]:
#Replace null values using when + otherwise

from pyspark.sql.functions import col, when

df_filled = df.withColumn(
    "age",
    when(col("age").isNull(), 0).otherwise(col("age"))
)


In [ ]:
#Replace NaN values with a constant

from pyspark.sql.functions import when, col, isnan

df_filled = df.withColumn(
    "age",
    when(col("age").isNull() | isnan(col("age")), 0).otherwise(col("age"))
)
#This replaces both null and NaN in the age column with 0.

In [ ]:
#Replace across multiple columns

from pyspark.sql.functions import when, col, isnan

for c in ["age", "salary"]:
    df = df.withColumn(
        c,
        when(col(c).isNull() | isnan(col(c)), 0).otherwise(col(c))
    )


In [ ]:
#Use na.fill() for null + na.replace() for NaN
# Replace nulls

df = df.na.fill({"age": 0, "salary": 0})

# Replace NaN values with 0

df = df.na.replace(float("nan"), 0)


In [ ]:
#One-liner function for any DataFrame

from pyspark.sql.functions import when, col, isnan

def replace_null_nan(df, replacements: dict):
    for c, val in replacements.items():
        df = df.withColumn(
            c,
            when(col(c).isNull() | isnan(col(c)), val).otherwise(col(c))
        )
    return df

# Example usage

df_filled = replace_null_nan(df, {"age": 0, "salary": 0, "city": "Unknown"})
